In [1]:
import json
import os
import re
import time
import chromadb

from dotenv import load_dotenv
from openai import OpenAI, RateLimitError
from sentence_transformers import SentenceTransformer

e:\BINUS_CODING\Ai Builders Hackhaton\backend\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_FILE = "data/startup-failure-insights.json"

EMBEDDING_MODEL_NAME = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

LOCAL_EMBEDDING_PATH = "models/multilingual-minilm"

CHROMA_PATH = "chroma_db"
COLLECTION_NAME = "startup_failures_v4"

OPENROUTER_MODEL = "inclusionai/ling-3.0-flash-fin:free"

REBUILD_DB = False

In [3]:
load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY tidak ditemukan di .env")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

print("OpenRouter client ready.")
print("Model:", OPENROUTER_MODEL)

OpenRouter client ready.
Model: inclusionai/ling-3.0-flash-fin:free


In [4]:
def call_llm(prompt, temperature = 0.0,max_retries = 5) :
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=OPENROUTER_MODEL,
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature = temperature
            )

            return (response.choices[0].message.content)

        except RateLimitError:
            wait_time = 5 * (attempt + 1)

            print(f"Rate limited.Retry in {wait_time}s...")

            time.sleep(wait_time)

    raise RuntimeError("Model tetap rate-limited setelah beberapa kali retry.")

In [5]:
#Testing the call_llm function
test_response = call_llm("Jawab persis dengan: call_llm berhasil.")

print(test_response)

call_llm berhasil.


In [6]:
if os.path.exists(LOCAL_EMBEDDING_PATH) :

    print("Loading local embedding model...")

    embedding_model = SentenceTransformer(LOCAL_EMBEDDING_PATH)

else :

    print("Downloading embedding model...")

    embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

    os.makedirs(
        os.path.dirname(LOCAL_EMBEDDING_PATH),
        exist_ok=True
    )

    embedding_model.save(
        LOCAL_EMBEDDING_PATH
    )

    print("Embedding model saved locally.")

print("Embedding model ready.")

Loading local embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2398.52it/s]


Embedding model ready.


In [7]:
with open(DATA_FILE,"r",encoding = "utf-8") as file:
    startups = json.load(file)

print("Total startup cases:", len(startups))

Total startup cases: 143


In [8]:
def create_embedding_document(startup) :

    return f"""
        Failure Question : {startup.get("prompt", "")}

        Historical Failure Reason : {startup.get("failure_reason", "")}
        """.strip()

In [9]:
chroma_client = (chromadb.PersistentClient(path=CHROMA_PATH))

print("ChromaDB ready.")

ChromaDB ready.


In [10]:
if REBUILD_DB:
    try:
        chroma_client.delete_collection(COLLECTION_NAME)

        print("Old collection deleted.")

    except Exception:
        print("No old collection found.")


collection = (
    chroma_client.get_or_create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"}
    )
)

print("Current documents:", collection.count())

Current documents: 143


In [11]:
if collection.count() == 0:
    print("Preparing documents...")

    documents = []
    ids = []
    metadatas = []

    for index, startup in enumerate(startups) :

        document = (create_embedding_document(startup))

        documents.append(document)

        ids.append(f"failure_{index}")

        metadatas.append(
            {
                "startup_index": index,

                "company":
                    startup.get("company",""),

                "funding":
                    startup.get("funding",""),

                "category":
                    startup.get("category","")
            }
        )

    print("Creating embeddings for", len(documents), "documents...")

    embeddings = (
        embedding_model.encode(
            documents,
            normalize_embeddings=True,
            show_progress_bar=True
        ).tolist()
    )

    collection.add(
        ids=ids,
        documents=documents,
        embeddings=embeddings,
        metadatas=metadatas
    )

    print("Vector database created : ",collection.count())

else:
    print("Vector database already exists:", collection.count())

Vector database already exists: 143


In [12]:
def parse_json_response(text):
    text = text.strip()

    text = re.sub(
        r"<think>.*?</think>",
        "",
        text,
        flags=(re.DOTALL | re.IGNORECASE)
    )

    text = text.strip()

    text = re.sub(
        r"^```json\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"^```\s*",
        "",
        text
    )

    text = re.sub(
        r"\s*```$",
        "",
        text
    )

    text = text.strip()

    try:
        return json.loads(text)

    except json.JSONDecodeError:

        start = text.find("{")
        end = text.rfind("}")

        if start == -1 or end == -1:
            raise ValueError("No valid JSON object found :\n\n"+ text)

        json_text = text[start:end + 1]

        return json.loads(json_text)

In [13]:
def analyze_plan(user_plan):

    prompt = f"""
        You are a decision stress-testing analyst.

        Your task is NOT to invent risks.

        Your task is to extract the most important
        assumptions that are directly supported by
        the user's plan.

        USER PLAN:
        {user_plan}

        Identify 3 to 5 material assumptions.

        For each assumption:

        1. source_evidence
        Quote or closely paraphrase the exact
        part of the plan that supports it.

        2. assumption
        State what the plan is implicitly or
        explicitly assuming must be true.

        3. failure_mechanism
        Explain how this assumption could fail.

        4. search_queries
        Generate exactly 3 English search queries
        for retrieving historical startup failure
        analogues.

        The 3 search queries should approach the
        same risk from different angles:

        - business or industry mechanics
        - causal failure mechanism
        - financial or operational mechanism

        IMPORTANT RULES:

        - Do not invent facts.
        - Do not infer unsupported causal links.
        - Limited capital does NOT automatically mean
        high customer acquisition cost.
        - Hiring many drivers does NOT automatically
        mean recruitment difficulty.
        - If early capacity is hired before sufficient
        demand exists, the relevant risk is
        underutilization or overcapacity.
        - Heavy discounts may imply subsidized growth,
        negative contribution margin, cash burn,
        or retention dependency.
        - Only include assumptions materially relevant
        to whether the plan succeeds or fails.

        Return ONLY valid JSON.

        Use exactly this schema:
        {{
            "assumptions": [
                {{
                    "source_evidence": "...",
                    "assumption": "...",
                    "failure_mechanism": "...",
                    "search_queries": [
                        "...",
                        "...",
                        "..."
                    ]
                }}
            ]
        }}
        """

    response = call_llm(prompt, temperature=0.0)

    return parse_json_response(response)

In [14]:
user_plan = """
    I want to launch an urban grocery delivery
    startup.

    For the grand opening, customers receive a
    50% discount on their first order for the
    first seven days.

    For the next three months, I will offer a
    20% weekend promotion.

    Customers who successfully refer three
    friends will receive one month of free
    delivery.

    I also plan to run offline promotional events
    at apartment complexes and university
    campuses.

    The goal is to grow customers quickly and
    build a strong ordering habit.

    I am willing to accept very low or even
    negative margins on promotional orders during
    the early growth phase.

    I expect customers acquired through these
    promotions to continue ordering after the
    discounts are reduced.

    I also plan to hire many delivery drivers
    early so that delivery times remain fast as
    orders grow.

    My initial capital is limited, so I expect
    order growth to eventually cover operating
    costs after several months.
    """

In [15]:
plan_analysis = analyze_plan(user_plan)

print(
    json.dumps(
        plan_analysis,
        indent=2,
        ensure_ascii=False
    )
)

{
  "assumptions": [
    {
      "source_evidence": "\"I expect customers acquired through these promotions to continue ordering after the discounts are reduced.\"",
      "assumption": "Customers acquired through heavy discounts will remain active and continue placing orders once promotional pricing is reduced or removed.",
      "failure_mechanism": "If acquired customers are primarily price-sensitive, they may churn as soon as discounts are reduced, meaning the startup bears the full cost of acquisition without recovering it through subsequent revenue. The plan's own goal of building a \"strong ordering habit\" would not materialize, and the promotional spending would yield no long-term customer base.",
      "search_queries": [
        "grocery delivery startup customer retention after promotional discounts end",
        "why discount-acquired customers churn when promotions end grocery delivery",
        "customer acquisition cost lifetime value grocery delivery startup failure"
 

In [16]:
def retrieve_multi_query(search_queries, n_results_per_query = 6) :
    merged = {}

    for query in search_queries:

        query_embedding = embedding_model.encode(
            query,
            normalize_embeddings=True
        ).tolist()

        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results_per_query
        )

        for metadata, document, distance in zip(
            results["metadatas"][0],
            results["documents"][0],
            results["distances"][0]
        ) :
            company = metadata["company"]
            distance = float(distance)

            if company not in merged :
                merged[company] = {
                    "company": company,
                    "funding": metadata["funding"],
                    "category": metadata["category"],
                    "startup_index": metadata["startup_index"],
                    "distance": distance,
                    "document": document,
                    "matched_queries": [query]
                }

            else :
                if query not in merged[company]["matched_queries"]:
                    merged[company]["matched_queries"].append(query)

                if distance < merged[company]["distance"]:
                    merged[company]["distance"] = distance

    candidates = list(merged.values())

    # deterministic retrieval pre-score
    for candidate in candidates:

        query_count = len(candidate["matched_queries"])

        # cosine distance:
        # makin kecil = makin mirip
        semantic_similarity = max(0, 1 - candidate["distance"])

        # max 3 search queries
        query_coverage = min(query_count / 3, 1)

        retrieval_score = (semantic_similarity * 0.75 + query_coverage * 0.25)

        candidate["retrieval_score"] = round(retrieval_score * 100, 2)

    candidates.sort(
        key=lambda x: x["retrieval_score"],
        reverse=True
    )

    return candidates

In [17]:
retrieval_results = []

for item in plan_analysis["assumptions"]:

    print("\nASSUMPTION:")

    print(item["assumption"])

    cases = retrieve_multi_query(
        item["search_queries"],
        n_results_per_query=6
    )

    # simpan top 5 kandidat
    cases = cases[:5]

    retrieval_results.append(
        {
            "assumption":
                item["assumption"],

            "source_evidence":
                item["source_evidence"],

            "failure_mechanism":
                item["failure_mechanism"],

            "search_queries":
                item["search_queries"],

            "cases":
                cases
        }
    )

    for case in cases:

        print(
            "-",
            case["company"], "| retrieval score:",

            case["retrieval_score"], "| queries:",

            len(case["matched_queries"]), "| distance:",
            
            round(case["distance"], 4)
        )


ASSUMPTION:
Customers acquired through heavy discounts will remain active and continue placing orders once promotional pricing is reduced or removed.
- Freshly | retrieval score: 63.36 | queries: 3 | distance: 0.4885
- BuyWithMe | retrieval score: 61.09 | queries: 3 | distance: 0.5188
- Zero Grocery | retrieval score: 55.0 | queries: 2 | distance: 0.489
- Fridge No More | retrieval score: 52.36 | queries: 2 | distance: 0.5241
- Munchery | retrieval score: 48.91 | queries: 2 | distance: 0.57

ASSUMPTION:
The rate of order growth will be sufficient to reach a point where revenue covers operating costs within the available capital runway.
- Zero Grocery | retrieval score: 71.73 | queries: 3 | distance: 0.3769
- PepperTap | retrieval score: 70.09 | queries: 3 | distance: 0.3988
- Freshly | retrieval score: 67.66 | queries: 3 | distance: 0.4312
- Fridge No More | retrieval score: 64.19 | queries: 3 | distance: 0.4775
- Sprig | retrieval score: 57.58 | queries: 2 | distance: 0.4544

ASSUMPT

In [18]:
def evaluate_candidate(assumption, failure_mechanism, candidate):

  prompt = f"""
    You are evaluating historical startup cases
    as analogical evidence for a decision
    stress test.

    CURRENT ASSUMPTION:
    {assumption}

    CURRENT FAILURE MECHANISM:
    {failure_mechanism}

    HISTORICAL CASE:
    Company:
    {candidate["company"]}

    Funding:
    {candidate["funding"]}

    Category:
    {candidate["category"]}

    Historical evidence:
    {candidate["document"]}

    Evaluate whether this historical case is
    actually useful evidence for the current risk.

    Score each dimension from 0 to 5:

    mechanism_match:
    Does the historical failure mechanism match
    the current risk?

    business_model_match:
    How similar are the operating/business
    mechanics?

    evidence_usefulness:
    How useful is this case for stress-testing
    the user's assumption?

    IMPORTANT:

    - Similar industry alone is not enough.
    - Similar wording alone is not enough.
    - A different industry can still be highly
      relevant if the causal mechanism matches.
    - Deep discounting + loss per order + cash burn
      strongly matches negative unit economics.
    - Expansion before sufficient demand may match
      overcapacity or underutilization.
    - Failure to reach sustainability before
      funding runs out may match runway risk.
    - Do not use information outside the supplied
      historical case.

    Return ONLY valid JSON:

    {{
        "mechanism_match": 0,
        "business_model_match": 0,
        "evidence_usefulness": 0,
        "reason": "..."
    }}
    """

  response = call_llm(prompt,temperature=0.0)

  return parse_json_response(response)

In [19]:
def calculate_relevance(evaluation):
    def clamp(value):

        return max(0, min(5, int(value)))

    mechanism = clamp(evaluation["mechanism_match"])

    business = clamp(evaluation["business_model_match"])

    usefulness = clamp(evaluation["evidence_usefulness"])

    weighted_score = (mechanism * 0.45 + usefulness * 0.35 + business * 0.20)

    score = round(weighted_score / 5 * 100)

    if score >= 75 :
        relevance = "HIGH"

    elif score >= 45 :
        relevance = "MEDIUM"

    else :
        relevance = "LOW"

    return (score,relevance)

In [20]:
reranked_results = []

for result in retrieval_results:
    print("\nEvaluating:", result["assumption"])

    evaluated_cases = []

    for candidate in result["cases"]:
        try:
            evaluation = (evaluate_candidate(result["assumption"],result["failure_mechanism"],candidate))

            (score,relevance) = calculate_relevance(evaluation)

            evaluated_cases.append(
            {
                "company":
                    candidate["company"],

                "funding":
                    candidate["funding"],

                "distance":
                    candidate["distance"],

                "retrieval_score":
                    candidate["retrieval_score"],

                "matched_query_count":
                    len(
                        candidate["matched_queries"]
                    ),

                "score": score,

                "relevance": relevance,

                "mechanism_match": evaluation["mechanism_match"],

                "business_model_match": evaluation["business_model_match"],

                "evidence_usefulness": evaluation["evidence_usefulness"],

                "reason": evaluation["reason"]
            }
)

        except Exception as e:

            print("Failed evaluating", candidate["company"], " : ", e)

    evaluated_cases.sort(
        key=lambda x: (
            x["score"],
            x["matched_query_count"]
        ),
        reverse=True
    )

    reranked_results.append(
        {
            "assumption":
                result[
                    "assumption"
                ],

            "source_evidence":
                result[
                    "source_evidence"
                ],

            "failure_mechanism":
                result[
                    "failure_mechanism"
                ],

            "cases":
                evaluated_cases
        }
    )


Evaluating: Customers acquired through heavy discounts will remain active and continue placing orders once promotional pricing is reduced or removed.

Evaluating: The rate of order growth will be sufficient to reach a point where revenue covers operating costs within the available capital runway.

Evaluating: Sufficient order volume will develop quickly enough to fully utilize the early fleet of hired delivery drivers, avoiding sustained idle capacity.

Evaluating: The combination of deep discounts and repeated promotions will successfully condition customers to form lasting ordering habits that persist at full margin.


In [21]:
for result in reranked_results:

    print("\n" + "=" * 80)

    print("ASSUMPTION:")

    print(result["assumption"])

    print("\nSOURCE EVIDENCE:")

    print(result["source_evidence"])

    print("\nFAILURE MECHANISM:")

    print(
        result[
            "failure_mechanism"
        ]
    )

    print(
        "\nHISTORICAL ANALOGUES:"
    )

    for case in result[
        "cases"
    ]:

        print(
            "\nCompany:",
            case["company"]
        )

        print("Retrieval score:", case["retrieval_score"])

        print("Score:", case["score"])

        print("Relevance:", case["relevance"])

        print("Mechanism:", case["mechanism_match"],"/ 5")

        print("Business model:",case["business_model_match"], "/ 5")

        print("Evidence usefulness:",case["evidence_usefulness"],"/ 5")

        print("Matched queries:",case["matched_query_count"])

        print("Vector distance:", round(case["distance"],4))

        print("Reason:", case["reason"])


ASSUMPTION:
Customers acquired through heavy discounts will remain active and continue placing orders once promotional pricing is reduced or removed.

SOURCE EVIDENCE:
"I expect customers acquired through these promotions to continue ordering after the discounts are reduced."

FAILURE MECHANISM:
If acquired customers are primarily price-sensitive, they may churn as soon as discounts are reduced, meaning the startup bears the full cost of acquisition without recovering it through subsequent revenue. The plan's own goal of building a "strong ordering habit" would not materialize, and the promotional spending would yield no long-term customer base.

HISTORICAL ANALOGUES:

Company: Zero Grocery
Retrieval score: 55.0
Score: 44
Relevance: LOW
Mechanism: 2 / 5
Business model: 3 / 5
Evidence usefulness: 2 / 5
Matched queries: 2
Vector distance: 0.489
Reason: Zero Grocery's failure was attributed to being 'chronically undercapitalized,' resulting in cash depletion and unpaid supplier debts wit

In [28]:
def generate_final_stress_test(
    user_plan,
    reranked_results
):
    final_results = []

    for result in reranked_results:

        strong_cases = [
            case
            for case in result["cases"]
            if case["relevance"] in ["HIGH", "MEDIUM"]
        ]

        historical_analogues = [
            {
                "company": case["company"],
                "score": case["score"],
                "relevance": case["relevance"],
                "reason": case["reason"]
            }
            for case in strong_cases[:3]
        ]

        evidence_json = json.dumps(
            historical_analogues,
            indent=2,
            ensure_ascii=False
        )

        prompt = f"""
You are an AI decision stress-testing analyst.

Analyze ONLY the single assumption provided below.

Do NOT use evidence from any other assumption.
Do NOT invent historical cases.
Do NOT introduce companies that are not included
in HISTORICAL ANALOGUES.

USER PLAN:
{user_plan}

ASSUMPTION:
{result["assumption"]}

SOURCE EVIDENCE:
{result["source_evidence"]}

FAILURE MECHANISM:
{result["failure_mechanism"]}

HISTORICAL ANALOGUES:
{evidence_json}

Your task is to create a practical pre-mortem
analysis for THIS assumption only.

Return:

1. risk
   Explain what could go wrong.

2. severity
   LOW, MEDIUM, or HIGH.

3. evidence
   Explain what the supplied historical
   analogues show.

   IMPORTANT:
   - If HISTORICAL ANALOGUES is empty,
     explicitly state that historical evidence
     is currently weak or unavailable.
   - Never introduce another company.
   - Historical cases are analogical evidence,
     not proof that the user's plan will fail.

4. what_must_be_true
   State what must be true for the assumption
   to succeed.

5. validation_experiment
   Suggest one cheap, concrete experiment that
   can be run before scaling.

6. validation_metric
   State what should be measured.

7. warning_signal
   State what observed result would indicate
   that the assumption may be unsafe.

IMPORTANT RULES:

- Do not predict an exact probability of failure.
- Do not invent historical facts.
- Do not use evidence outside the provided
  historical analogues.
- Do not invent numeric thresholds.
- Do not propose percentage, count, frequency,
  or time thresholds unless they are explicitly
  provided in the user plan or historical evidence.
- Numeric decision thresholds will be calculated
  separately by the quantitative stress-testing engine.
- Prefer measurable validation experiments.
- Do not recommend scaling before validation.

Return ONLY valid JSON.

Schema:

{{
    "assumption": "...",
    "risk": "...",
    "severity": "LOW|MEDIUM|HIGH",
    "evidence": "...",
    "what_must_be_true": "...",
    "validation_experiment": "...",
    "validation_metric": "...",
    "warning_signal": "..."
}}
"""

        response = call_llm(
            prompt,
            temperature=0.0
        )

        analysis = parse_json_response(
            response
        )

        final_results.append(
            analysis
        )

    return {
        "stress_test": final_results
    }

In [29]:
final_stress_test = generate_final_stress_test(
    user_plan,
    reranked_results
)

print(
    json.dumps(
        final_stress_test,
        indent=2,
        ensure_ascii=False
    )
)

{
  "stress_test": [
    {
      "assumption": "Customers acquired through heavy discounts will remain active and continue placing orders once promotional pricing is reduced or removed.",
      "risk": "If acquired customers are primarily price-sensitive, they may churn as soon as discounts are reduced, meaning the startup bears the full cost of acquisition without recovering it through subsequent revenue. The plan's own goal of building a 'strong ordering habit' would not materialize, and the promotional spending would yield no long-term customer base.",
      "severity": "HIGH",
      "evidence": "Historical evidence is currently weak or unavailable. No historical analogues were provided to support or refute this assumption.",
      "what_must_be_true": "Customers must derive sufficient ongoing value from the service beyond price — such as convenience, product quality, or reliability — to continue ordering once promotional pricing ends. The service must also be functional and competi